# The Heart of vLLM: Request Orchestration with EngineCore

In the last notebook, we saw how users interact with vLLM through its high-level API. But what happens once a request enters the system? How does it go from a simple text prompt to a stream of generated tokens?

This notebook dives into `EngineCore`, the central component that acts as the brain of the vLLM system. It's the master coordinator, orchestrating the entire inference lifecycle from the moment a request is received to the moment its output is ready. By the end of this notebook, you will be able to describe the role of EngineCore as the central coordinating entity and understand how it manages the flow of requests between the scheduling and execution components.

In [ ]:
# === Setup Cell ===
# All imports needed for the rest of the notebook.

from collections import deque
from typing import List, Dict, Any

# For visualization
import graphviz

# A simple dataclass to represent a user request
class Request:
    def __init__(self, request_id: str, prompt: str, max_tokens: int):
        self.request_id = request_id
        self.prompt = prompt
        self.max_tokens = max_tokens
        self.output_tokens = []
        self.is_finished = False
        print(f"  [Request {request_id} created for prompt: '{prompt[:20]}...']")

    def append_token(self, token: str):
        self.output_tokens.append(token)
        if len(self.output_tokens) >= self.max_tokens:
            self.is_finished = True

    def __repr__(self):
        return (f"Request(id={self.request_id}, "
                f"output='{''.join(self.output_tokens)}', "
                f"finished={self.is_finished})")

# Define simple types for clarity
ScheduledBatch = List[Request]
ModelOutput = Dict[str, str] # Maps request_id to a new token
FinalOutput = Dict[str, str] # Maps request_id to a new token

### The Core Idea: An Air Traffic Controller

Before we dive into code, let's use an analogy. Imagine a busy airport. You have planes arriving (requests), runways (GPU resources), and a complex schedule to manage.

- **The Scheduler** is like the airport's planning department. It looks at all the planes waiting to take off, considers their priority and destination, and creates a plan for which planes should go to which runway *in the next few minutes*. It's focused on planning and resource allocation (i.e., assigning KV cache blocks).

- **The Executor** is like the ground crew and the runway itself. It takes a specific, concrete instruction—"Plane A77, take off from Runway 3"—and executes it. It handles the physics of the takeoff, communicates with the plane, and confirms when it's airborne. It's focused on the physical execution (running the model on the GPU).

But who connects the planners to the ground crew? Who manages the real-time, step-by-step operation? That's the **Air Traffic Controller**.

This is the role of `EngineCore`.

`EngineCore` runs a continuous loop. In each "step" of the loop, it:
1.  Asks the `Scheduler` for the next batch of requests to process.
2.  Hands this batch to the `Executor` to run the actual model inference.
3.  Receives the results (the newly generated tokens) from the `Executor`.
4.  Gives these results back to the `Scheduler` so it can update the state of each request (e.g., mark it as finished, or ready for the next token).
5.  Repeats.

`EngineCore` doesn't do the detailed planning or the heavy lifting itself. Instead, it serves as the essential, high-level coordinator, ensuring that the planning (scheduling) and execution phases work together seamlessly in a tight loop.



Now, let's build a simplified version of this coordinator.

In [ ]:
# === Minimal Implementation ===
# These are mock versions of the real, complex vLLM components.

class MockScheduler:
    """A simplified scheduler that manages request queues."""
    def __init__(self):
        self.waiting_queue = deque()
        self.running_map = {}

    def add_request(self, request: Request):
        self.waiting_queue.append(request)

    def has_requests(self) -> bool:
        return bool(self.waiting_queue) or bool(self.running_map)

    def schedule(self) -> ScheduledBatch:
        # For simplicity, schedule one request from waiting to running.
        if self.waiting_queue:
            request = self.waiting_queue.popleft()
            self.running_map[request.request_id] = request
            print(f"[Scheduler] Scheduled request {request.request_id}. Running: {list(self.running_map.keys())}")
            return [request]
        return []

    def update_from_output(self, batch: ScheduledBatch, model_output: ModelOutput) -> FinalOutput:
        final_outputs = {}
        for request in batch:
            new_token = model_output.get(request.request_id)
            if new_token:
                request.append_token(new_token)
                final_outputs[request.request_id] = new_token
                if request.is_finished:
                    print(f"[Scheduler] Request {request.request_id} finished.")
                    del self.running_map[request.request_id]
        return final_outputs


class MockExecutor:
    """A simplified executor that "runs" the model."""
    def execute_model(self, batch: ScheduledBatch) -> ModelOutput:
        print(f"[Executor]  Executing model for {len(batch)} request(s)...")
        # Simulate generating one token for each request in the batch.
        return {req.request_id: "a" for req in batch}


class EngineCore:
    """The central coordinator."""
    def __init__(self, scheduler: MockScheduler, executor: MockExecutor):
        self.scheduler = scheduler
        self.executor = executor

    def step(self) -> (FinalOutput, bool):
        """Performs one iteration of scheduling, execution, and updating."""
        if not self.scheduler.has_requests():
            return {}, False

        # 1. Get the next batch of work from the scheduler.
        scheduled_batch = self.scheduler.schedule()
        if not scheduled_batch:
            return {}, False # Nothing to execute

        # 2. Execute the model with the batch.
        model_output = self.executor.execute_model(scheduled_batch)

        # 3. Update scheduler state with the results.
        final_outputs = self.scheduler.update_from_output(scheduled_batch, model_output)

        return final_outputs, True

### Implementation Walk-through

Let's break down the `EngineCore` class. It's surprisingly simple because its primary job is delegation.

**`__init__(self, scheduler: MockScheduler, executor: MockExecutor)`**
```python
def __init__(self, scheduler: MockScheduler, executor: MockExecutor):
    self.scheduler = scheduler
    self.executor = executor
```
The constructor is straightforward. It takes instances of a `Scheduler` and an `Executor`. This is a great example of the "dependency injection" pattern: `EngineCore` doesn't create its collaborators; it's given them. This makes the system modular and easy to test.

**`step(self)`**
This method represents a single cycle of the inference loop.

```python
# 1. Get the next batch of work from the scheduler.
scheduled_batch = self.scheduler.schedule()
```
First, it calls `self.scheduler.schedule()`. This is `EngineCore` asking the planning department, "What's the plan for the next execution cycle?" The `Scheduler` returns a `ScheduledBatch`, which is just a list of `Request` objects that have been cleared for execution (e.g., they have been allocated KV cache blocks).

```python
# 2. Execute the model with the batch.
model_output = self.executor.execute_model(scheduled_batch)
```
Next, `EngineCore` hands this concrete plan to the `Executor` by calling `self.executor.execute_model()`. This is the "go" command. The `Executor` does the heavy lifting of running the model and returns the result—a dictionary mapping each request ID to its newly generated token.

```python
# 3. Update scheduler state with the results.
final_outputs = self.scheduler.update_from_output(scheduled_batch, model_output)
```
Finally, `EngineCore` takes the results from the `Executor` and reports back to the `Scheduler` by calling `self.scheduler.update_from_output()`. This "closes the loop," allowing the `Scheduler` to update its internal state. For example, it appends the new token to the request's output and checks if the request has now met its `max_tokens` limit. If so, the `Scheduler` can mark it as finished and free up its resources for other requests.

The `step` method then returns the generated outputs, which would eventually be sent back to the user.

In [ ]:
# === Demonstration ===

# 1. Set up the components
mock_scheduler = MockScheduler()
mock_executor = MockExecutor()
engine_core = EngineCore(mock_scheduler, mock_executor)

# 2. Add some requests to the system
print("--- Adding Requests ---")
req1 = Request(request_id="req-001", prompt="The capital of France is", max_tokens=3)
req2 = Request(request_id="req-002", prompt="The opposite of hot is", max_tokens=4)
mock_scheduler.add_request(req1)
mock_scheduler.add_request(req2)
print("-" * 25)

# 3. Run the EngineCore's main loop step-by-step
step_count = 0
while mock_scheduler.has_requests():
    step_count += 1
    print(f"\n--- EngineCore Step {step_count} ---")
    outputs, executed = engine_core.step()
    if executed:
        print(f"  [EngineCore] Step produced outputs: {outputs}")
    else:
        print("  [EngineCore] No batch was executed in this step.")

# 4. Final state of requests
print("\n--- Final Request States ---")
print(req1)
print(req2)

### Going Deeper: `step` vs. `step_with_batch_queue`

In the real vLLM source code, you'll notice two step functions: `step` and `step_with_batch_queue`. Why the complexity?

This is an optimization for a technique called **pipeline parallelism**, where different stages of the model (e.g., different layers) are run on different GPUs.

Imagine an assembly line with two stations.
- Station 1 (`schedule` + `execute_model`)
- Station 2 (`sample_tokens`)

With the simple `step` function, the process looks like this:
1. Station 1 processes a batch. Station 2 is **idle**.
2. Station 1 is **idle**. Station 2 processes the batch.

The idle time is called a "pipeline bubble" and it reduces overall throughput.

The `step_with_batch_queue` method solves this. It creates a queue of work between the stations.

1.  **Step 1:** `EngineCore` schedules Batch A and submits it for execution. It immediately returns without waiting for the result.
2.  **Step 2:** While the GPU is busy executing Batch A, `EngineCore` schedules Batch B. Now the CPU (scheduling) and GPU (executing) are working in parallel.
3.  **Step 3:** The result for Batch A is ready. `EngineCore` processes it. At the same time, it submits Batch B for execution.

By creating a small buffer (`batch_queue`) of scheduled work, `EngineCore` ensures that the executor (the GPU) is almost never idle. It can immediately pick up the next batch from the queue as soon as it finishes the current one. This keeps the "assembly line" full and maximizes the system's throughput, even though it might add a tiny bit of latency to any single request.

In [ ]:
# === Visualization ===

# Let's visualize the flow of data and control within the vLLM engine.
# This diagram shows how EngineCore acts as the central coordinator.

dot = graphviz.Digraph('EngineCore', comment='The vLLM EngineCore Architecture')
dot.attr('node', shape='box', style='rounded,filled', fillcolor='lightblue')
dot.attr(rankdir='TB', label='EngineCore Orchestration Flow', labelloc='t', fontsize='20')

with dot.subgraph(name='cluster_user') as c:
    c.attr(style='filled', color='lightgrey')
    c.node_attr.style = 'filled'
    c.attr(label='User-Facing Layer')
    c.node('LLMEngine', 'LLMEngine (API)')

with dot.subgraph(name='cluster_core') as c:
    c.attr(color='blue')
    c.attr(label='Core Components')
    c.node('EngineCore', 'EngineCore\n(The Coordinator)')
    c.node('Scheduler', 'Scheduler\n(Planning & State)')
    c.node('Executor', 'Executor\n(GPU Execution)')

# Edges showing the flow
dot.edge('LLMEngine', 'EngineCore', label='add_request()')

# The core step loop
dot.edge('EngineCore', 'Scheduler', label='1. schedule()', style='dashed', constraint='false')
dot.edge('Scheduler', 'EngineCore', label='ScheduledBatch', style='dashed')
dot.edge('EngineCore', 'Executor', label='2. execute_model()', style='dashed')
dot.edge('Executor', 'EngineCore', label='ModelOutput', style='dashed')
dot.edge('EngineCore', 'Scheduler', label='3. update_from_output()', style='dashed', constraint='false')
dot.edge('EngineCore', 'LLMEngine', label='Final Outputs')


# Render the graph
dot

### Summary: The Grand Central Station

In this notebook, we built a simplified model of vLLM's `EngineCore` and saw it in action. We learned that while other components handle the complex details of scheduling algorithms or low-level GPU operations, `EngineCore` plays the crucial role of the master coordinator.

**Key Takeaways:**
*   **Central Coordinator:** `EngineCore` is the heart of the vLLM engine, orchestrating the interaction between the `Scheduler` and the `Executor`.
*   **The `step` Loop:** The core of its operation is a simple but powerful loop: schedule, execute, update. This loop drives the entire inference process forward, one token at a time for all concurrent requests.
*   **Delegation over Implementation:** `EngineCore`'s logic is minimal because it delegates tasks. It trusts the `Scheduler` to make smart plans and the `Executor` to carry them out efficiently.
*   **Optimizations for Throughput:** Advanced features like the `batch_queue` show how this simple coordination model can be extended to hide latency and maximize hardware utilization in parallel environments.

**What's Next?**

We've mentioned the `Scheduler` many times as the "planner." But how does it actually make its decisions? How does it choose which requests to run next, and how does it manage the precious KV cache memory? Our next notebook, **Efficient Request Handling: The Scheduler**, will dive deep into these questions.